In [1]:
import sys

print(sys.executable)

c:\Users\lizcr\OneDrive\Documents\MSc\Project\msc_project\.venv\Scripts\python.exe


In [1]:
import copy, math, os, pickle, time, pandas as pd, numpy as np, scipy.stats as ss

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, accuracy_score, f1_score

import torch, torch.utils.data as utils, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.autograd import Variable
from torch.nn.parameter import Parameter

In [2]:
GAP_TIME          = 6  # In hours
WINDOW_SIZE       = 24 # In hours
SEED              = 1
ID_COLS           = ['subject_id', 'hadm_id', 'icustay_id']
ID_COLS_HOURLY = ID_COLS + ['hours_in']
TESTING = True          # set False for the full cohort run

np.random.seed(SEED)
torch.manual_seed(SEED)

In [3]:
class DictDist():
    def __init__(self, dict_of_rvs): self.dict_of_rvs = dict_of_rvs
    def rvs(self, n):
        a = {k: v.rvs(n) for k, v in self.dict_of_rvs.items()}
        out = []
        for i in range(n): out.append({k: vs[i] for k, vs in a.items()})
        return out
    
class Choice():
    def __init__(self, options): self.options = options
    def rvs(self, n): return [self.options[i] for i in ss.randint(0, len(self.options)).rvs(n)]

In [4]:
import getpass
from sqlalchemy import create_engine

pg_user = 'postgres'      # same value you use to connect via psql
pg_host = 'localhost'          # or wherever your Postgres server is
pg_port = 5432
pg_dbname = 'mimiciv'

pg_password = getpass.getpass('Postgres password: ')

engine = create_engine(
    f'postgresql+psycopg2://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_dbname}'
)

In [5]:
df = pd.read_sql("SELECT current_database();", engine)
print(df)

  current_database
0          mimiciv


In [6]:
pd.read_sql('SELECT 1', engine)

,?column?
0,1


In [7]:
%%time

hourly_table = 'msc_project.sample_hourly_data' if TESTING else 'msc_project.hourly_data'
statics_table = 'msc_project.sample_allpatients' if TESTING else 'msc_project.allpatients'

hourly_query = f"SELECT * FROM {hourly_table}"
statics_query = f"SELECT * FROM {statics_table}"

data_full_lvl2 = pd.read_sql(hourly_query, engine).set_index(ID_COLS_HOURLY)

statics = pd.read_sql(statics_query, engine).set_index(ID_COLS)

CPU times: total: 62.5 ms
Wall time: 69.6 ms


In [8]:
data_full_lvl2.head()

hour_end  heart_rate  \
subject_id hadm_id  icustay_id hours_in                                   
12894275   20050533 30397733   24       2157-02-26 03:00:00        88.0   
                               23       2157-02-26 02:00:00        85.0   
                               22       2157-02-26 01:00:00        99.0   
                               21       2157-02-26 00:00:00        90.0   
                               20       2157-02-25 23:00:00        89.0   

                                           sbp   dbp   mbp  sbp_ni  dbp_ni  \
subject_id hadm_id  icustay_id hours_in                                      
12894275   20050533 30397733   24        119.0  63.0  74.0   119.0    63.0   
                               23        114.0  63.0  74.0   114.0    63.0   
                               22        117.0  67.0  78.0   117.0    67.0   
                               21        116.0  54.0  68.0   116.0    54.0   
                               20        120.0  68.5  76.5   120.0    68.5   

                                         mbp_ni  temperature  spo2  ...   gcs  \
subject_id hadm_id  icustay_id hours_in                             ...         
12894275   20050533 30397733   24          74.0          NaN  98.0  ...   NaN   
                               23          74.0          NaN  99.0  ...   NaN   
                               22          78.0          NaN  98.0  ...   NaN   
                               21          68.0        37.17  98.0  ...  15.0   
                               20          76.5          NaN  99.0  ...   NaN   

                                         hematocrit hemoglobin mch  mchc  mcv  \
subject_id hadm_id  icustay_id hours_in                                         
12894275   20050533 30397733   24               NaN        NaN NaN   NaN  NaN   
                               23               NaN        NaN NaN   NaN  NaN   
                               22               NaN        NaN NaN   NaN  NaN   
                               21               NaN        NaN NaN   NaN  NaN   
                               20               NaN        NaN NaN   NaN  NaN   

                                         platelet  rbc  rdw  wbc  
subject_id hadm_id  icustay_id hours_in                           
12894275   20050533 30397733   24             NaN  NaN  NaN  NaN  
                               23             NaN  NaN  NaN  NaN  
                               22             NaN  NaN  NaN  NaN  
                               21             NaN  NaN  NaN  NaN  
                               20             NaN  NaN  NaN  NaN  

[5 rows x 34 columns]

In [9]:
statics.head()

,,,gender,dod,admittime,dischtime,admission_type,admission_location,admission_age,race,hospital_expire_flag,los_icu
subject_id,hadm_id,icustay_id,,,,,,,,,,
15386345,20490942,32585414,M,2160-02-07,2156-09-30 16:31:00,2156-10-14 14:45:00,EW EMER.,WALK-IN/SELF REFERRAL,71,WHITE,0,1.08
18583067,25650481,31835914,M,2125-05-31,2124-12-26 16:27:00,2125-01-07 20:11:00,EW EMER.,EMERGENCY ROOM,52,WHITE,0,1.96
17122654,27905430,35106498,F,2170-10-20,2170-09-03 19:04:00,2170-09-16 14:35:00,URGENT,TRANSFER FROM HOSPITAL,83,WHITE,0,5.00
10544642,27891616,37540625,F,None,2126-06-21 13:28:00,2126-06-26 18:05:00,EW EMER.,PROCEDURE SITE,70,WHITE,0,3.88
11001569,23809683,30851202,F,2146-06-29,2146-06-20 16:29:00,2146-06-29 03:46:00,OBSERVATION ADMIT,WALK-IN/SELF REFERRAL,88,WHITE - RUSSIAN,1,6.00


In [ ]:
""""
Adapted from the MIMIC-Extract function 'simple_imputer' in mimic3benchmark.preprocessing.utils, which
deals with missing values in the hourly data. The original function takes a dataframe with a multi-index of 
(subject_id, hadm_id, icustay_id, hours_in) and columns with a multi-index of (label, LEVEL1, LEVEL2, 
Aggregation Function). The function fills in missing values for the 'mean' aggregation function using 
forward fill and the mean of the icustay, and creates a 'mask' column indicating whether the original value 
was present or not. It also calculates the time since the last measurement for each variable.

This adaptation takes a dataframe with only one column per variable for the mean per hour. This is indexed by 
the ID columns and hour. If there is no measurement in a given hour, it is supplied as NaN in the input. 
"""

def simple_imputer(df, id_cols=ID_COLS):

    df = df.copy()
    # NB the forward fill depends on hours_in being in order for each stay.
    df = df.sort_index()    # Therefore sort. The index has been set earlier to be the ID_COLS + ['hours_in'] for the hourly data.

    mask = df.notna().astype(float)    # see later for why we convert to float. This is the mask of whether a 
                                        # measurement was present or not.
    imputed = (
        df.groupby(level=id_cols).ffill()              # forward fills from the last measurement.
        .fillna(0)
    )

    # Grouping by stay here means that the last observed time is only within the stay, 
    # not across other stays as in the original notebook. 
    is_absent = 1 - mask
    hours_of_absence = is_absent.groupby(level=id_cols).cumsum()

    df_out = pd.concat(
        [imputed, mask, hours_of_absence],
        axis = 1,
        keys = ['imputed', 'mask', 'hours_of_absence']
    )
    return df_out

In [15]:
simple_imputer(data_full_lvl2).head()

imputed                  \
                                                   hour_end heart_rate  sbp   
subject_id hadm_id  icustay_id hours_in                                       
10164613   22813323 39738251   -24      2171-12-16 15:00:00        0.0  0.0   
                               -23      2171-12-16 16:00:00        0.0  0.0   
                               -22      2171-12-16 17:00:00        0.0  0.0   
                               -21      2171-12-16 18:00:00        0.0  0.0   
                               -20      2171-12-16 19:00:00        0.0  0.0   

                                                                        \
                                         dbp  mbp sbp_ni dbp_ni mbp_ni   
subject_id hadm_id  icustay_id hours_in                                  
10164613   22813323 39738251   -24       0.0  0.0    0.0    0.0    0.0   
                               -23       0.0  0.0    0.0    0.0    0.0   
                               -22       0.0  0.0    0.0    0.0    0.0   
                               -21       0.0  0.0    0.0    0.0    0.0   
                               -20       0.0  0.0    0.0    0.0    0.0   

                                                          ...  \
                                        temperature spo2  ...   
subject_id hadm_id  icustay_id hours_in                   ...   
10164613   22813323 39738251   -24              0.0  0.0  ...   
                               -23              0.0  0.0  ...   
                               -22              0.0  0.0  ...   
                               -21              0.0  0.0  ...   
                               -20              0.0  0.0  ...   

                                        hours_of_absence             \
                                                     gcs hematocrit   
subject_id hadm_id  icustay_id hours_in                               
10164613   22813323 39738251   -24                   1.0        1.0   
                               -23                   2.0        2.0   
                               -22                   3.0        3.0   
                               -21                   4.0        4.0   
                               -20                   5.0        5.0   

                                                                            \
                                        hemoglobin  mch mchc  mcv platelet   
subject_id hadm_id  icustay_id hours_in                                      
10164613   22813323 39738251   -24             1.0  1.0  1.0  1.0      1.0   
                               -23             2.0  2.0  2.0  2.0      2.0   
                               -22             3.0  3.0  3.0  3.0      3.0   
                               -21             4.0  4.0  4.0  4.0      4.0   
                               -20             5.0  5.0  5.0  5.0      5.0   

                                                        
                                         rbc  rdw  wbc  
subject_id hadm_id  icustay_id hours_in                 
10164613   22813323 39738251   -24       1.0  1.0  1.0  
                               -23       2.0  2.0  2.0  
                               -22       3.0  3.0  3.0  
                               -21       4.0  4.0  4.0  
                               -20       5.0  5.0  5.0  

[5 rows x 102 columns]